# 6. Supersegments and access prediction

Given one catheter pathway, how hard will it be to reach the target? ArterialGNet answers with a probability of difficult access, averaged over five models, plus an attention map that shows which part of the pathway drove the answer.

This uses the supersegments from notebook 4 and runs in seconds on a GPU.

In [ ]:
import os, numpy as np, torch, matplotlib.pyplot as plt
from arterial_nb import *
from arterial import model_registry
from arterial.io.load_and_save_operations import load_json
from arterial.access_prediction.access_predictor import AccessPredictor

predictor = AccessPredictor(case("access"), fixture("cta.nii.gz"), supersegments_dir_path=derived("supersegments"), access=["femoral"], side=["left", "right"])
with timed("preprocessing"), quiet():
    predictor.preprocess_supersegments(save=True)
plt.close("all")   # the stage saves its own plots to disk; no need to show them here

## What the model sees

Each pathway is turned into two graphs plus a short vector. The **dense graph** is the pathway itself, one node every 2 mm with its local features. The **segment graph** has one node per named vessel with the segment features. The **global vector** holds the arch type and a few pathway-level numbers. Everything is normalised with the statistics stored next to the weights.

In [ ]:
for key, pre in predictor.preprocessed_supersegment_dict.items():
    print(f"{' + '.join(key)}: dense graph {pre['dense_graph'].number_of_nodes()} nodes, segment graph {pre['segment_graph'].number_of_nodes()} nodes, {len(pre['global_features']['features_list'])} global features")

In [ ]:
description = load_json(model_registry.model_path("access_prediction", "dataset.json"))
print("global features:", ", ".join(description["global_feature_names"]))

## The network

The checkpoint stores the architecture it was trained with. The dense path is a graph attention layer over the pathway, pooled into one vector; the output layer turns it into two class probabilities. The attention weights of that first layer are what the attention map shows.

In [ ]:
state = torch.load(model_registry.model_path("access_prediction", "fold_0", "model_weights.pth"), map_location="cpu", weights_only=False)
kwargs = {k: v for k, v in state["init_kwargs"].items() if not callable(v) and k not in ("training", "global_path", "segment_path")}
print(", ".join(f"{k}={v}" for k, v in kwargs.items()))

In [ ]:
with timed("prediction, 5 folds, both sides"), quiet():
    predictor.predict_accessibility(return_attention_map=True, save=True)
for k, v in predictor.predictions_dict.items():
    print(f"{' + '.join(k)}: P(difficult access) = {float(v['mean']):.2f}  (spread across folds {float(v['std']):.2f})")

## Attention map

Every edge of the pathway gets the attention the network paid to it, averaged over heads and folds; nodes take the mean of their edges. High attention marks the region that made the pathway look difficult. The same map is written as a VTK file and a JSON of points so it can be opened in 3D Slicer next to the CTA.

In [ ]:
fig = plt.figure(figsize=(16, 9))
for idx, (key, graph) in enumerate(predictor.attention_maps_dict.items()):
    plot_graph(graph, color_by=lambda d: d["features femoral"]["attention_weight"], ax=fig.add_subplot(1, 2, idx + 1, projection="3d"),
               title=f"{' + '.join(key)}: P = {predictor.predictions_dict[key]['mean']:.2f}", node_size=12)

In [ ]:
out_dir = predictor.access_prediction_dir_path
for r, _, fs in os.walk(out_dir):
    for f in sorted(fs):
        print(os.path.relpath(os.path.join(r, f), out_dir))

## Notes

- The spread across the five folds is the model's own uncertainty; a large spread with a middling probability is a case to review by eye.
- Only the femoral anterior pathways have a trained model today; radial and posterior supersegments are produced but not scored.
- A pathway that collapses to a single segment (for example when labelling failed upstream) is rejected with a clear error rather than scored.